# Step-by-step diagnostic: what the model sees, says and how it is graded

Every stage of the baseline pipeline is shown and checked on its own:

| step | what you see | what it rules out |
|---|---|---|
| 1 | the raw dataset items | wrong data loaded |
| 2 | **independent SymPy check of every gold answer** (from the text only) | wrong gold labels |
| 3 | model, tokenizer, the token ids used for A / B / C | scoring the wrong tokens |
| 4 | **the exact model input**, special tokens included, token by token | broken chat template |
| 5 | next-token distribution per item: top-10 tokens, A/B/C log-probs, prediction | scoring bug |
| 6 | the same items through the production script (batched) vs. step 5 (one by one) | batching / padding bug |
| 7 | **the model's free-text answer** (greedy generation) vs. its letter score | "the model meant something else" |
| 8 | exactly what is passed to the grader, and the grader's result | grading bug |
| 9 | easy control questions in the same format | the model cannot do the format at all |
| 10 | options re-ordered: does the choice follow the content or the letter? | letter bias vs. reasoning |

Needs on Drive: `baseline_letter_logprobs.py`, `gold_check.py`, `ivp_mcq/`. T4 GPU runtime.

In [1]:
# --- setup -------------------------------------------------------------------
import os, sys, json, math, subprocess, glob, re
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/Thesis'          # <- edit if needed
    os.chdir(PROJECT_DIR)
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF_TOKEN set')
    except Exception as e:
        print('no HF_TOKEN secret:', type(e).__name__)
sys.path[:0] = [os.getcwd(), os.path.join(os.getcwd(), 'ivp_mcq')]
import torch, transformers, pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 250)
print('cwd', os.getcwd()); print('torch', torch.__version__, '| transformers', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

Mounted at /content/drive
HF_TOKEN set
cwd /content/drive/MyDrive/Thesis
torch 2.11.0+cu130 | transformers 5.21.0
GPU: Tesla T4


In [2]:
# --- settings ----------------------------------------------------------------
MODEL_KEY  = 'llama-3.2-3b-instruct'   # any key of baseline_letter_logprobs.MODELS
SPLIT      = 'dev'
SYSTEM     = None                      # None = prompt v1, or 'letter-only'
N_SHOW     = 6                         # items printed in full (3 B + 3 C)
GEN_TOKENS = 400                       # max new tokens for the free-text answers (step 7)

## Step 1 - the dataset items

In [3]:
from grade_mcq import load, evaluate, extract_letter, letter_from_scores, LETTERS
items = load(f'ivp_mcq/ivp_mcq_{SPLIT}.jsonl')
B = [i for i in items if i['variant'] == 'B']; C = [i for i in items if i['variant'] == 'C']
show = B[:N_SHOW // 2] + C[:N_SHOW - N_SHOW // 2]
print(f'{len(items)} items in {SPLIT}; gold letters', {L: sum(i["answer"] == L for i in items) for L in LETTERS})
for it in show:
    print('\n' + '=' * 100); print(it['id'], '| variant', it['variant'], it['subtype'], '|', it['ode_type'], '| gold', it['answer'])
    print('-' * 100); print(it['prompt']); print('-' * 100)
    for k, c in enumerate(it['conditions'], 1):
        print(f'  condition {k}: {c["text"]:30s} role={c["role"]}')
    for o in it['options']:
        print(f'  option {o["letter"]}: {o["text"]:40s} role={o["role"]:10s} satisfies conditions {o["satisfies_conditions"]}')

72 items in dev; gold letters {'A': 29, 'B': 25, 'C': 18}

ivp-0000 | variant B unrelated | first_order_direct | gold A
----------------------------------------------------------------------------------------------------
Solve the initial value problem below and choose the correct solution.

Differential equation: y' = -exp(x)
Conditions:
1. y(1) = -4 - exp(1)
2. z(1) = -exp(1) - 1

Options:
A) y = -exp(x) - 4
B) y = -exp(x) - 1
C) y = 1 - exp(x)
Answer with the letter of the correct option only.
----------------------------------------------------------------------------------------------------
  condition 1: y(1) = -4 - exp(1)             role=sufficient
  condition 2: z(1) = -exp(1) - 1             role=distractor
  option A: y = -exp(x) - 4                          role=correct    satisfies conditions [1]
  option B: y = -exp(x) - 1                          role=trap       satisfies conditions []
  option C: y = 1 - exp(x)                           role=distractor satisfies conditi

## Step 2 - are the gold answers correct?
`gold_check.py` ignores the labels: it substitutes each option into the ODE and into
each condition with SymPy, derives the answer itself, and compares it with the gold
letter and with the dataset metadata.

In [4]:
from gold_check import check_item, show as show_check
checks = [check_item(it) for it in items]
for it in show:
    show_check(it, check_item(it))
n_ok = sum(c['gold_ok'] and c['meta_ok'] and c['all_solve_ode'] for c in checks)
print(f'\nALL {len(items)} {SPLIT} items: {n_ok} OK (gold re-derived, every option solves the ODE, metadata matches)')
print('mismatches:', [c['id'] for c in checks if not c['gold_ok']] or 'none')
print('grader on the gold letters themselves:', evaluate(items, [i['answer'] for i in items])[0])


=== ivp-0000  variant B:unrelated  first_order_direct
  equation: y' = -exp(x)
  cond 1: y(1) = -4 - exp(1)               [sufficient]
  cond 2: z(1) = -exp(1) - 1               [distractor]
  opt text                                    solves ODE  satisfies (mine)  determines y  metadata  role
  A   y = -exp(x) - 4                         True        [1]               True          [1]       correct
  B   y = -exp(x) - 1                         True        []                False         []        trap
  C   y = 1 - exp(x)                          True        []                False         []        distractor
  needs all 1 y-conditions, determining -> derived answer A, gold A  OK

=== ivp-0001  variant B:unrelated  first_order_direct
  equation: y' = -3*exp(x)
  cond 1: z(-1) = 3 - 3*exp(-1)            [distractor]
  cond 2: y(-1) = -3*exp(-1) - 1           [sufficient]
  opt text                                    solves ODE  satisfies (mine)  determines y  metadata  role
  A   y 

## Step 3 - model, tokenizer and the answer-letter tokens

In [5]:
from baseline_letter_logprobs import (MODELS, LETTER_FORMS, TEMPLATE_KWARGS, LOAD_KWARGS, GPU_DTYPE,
                                      SYSTEM_PRESETS, build_text, letter_ids, supports_system,
                                      model_revision, dtype_kwarg)
from transformers import AutoModelForCausalLM, AutoTokenizer
hf_name, fmt = MODELS[MODEL_KEY]
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = {'fp16': torch.float16, 'fp32': torch.float32}[GPU_DTYPE.get(MODEL_KEY, 'fp16') if device == 'cuda' else 'fp32']
tok = AutoTokenizer.from_pretrained(hf_name)
model = AutoModelForCausalLM.from_pretrained(hf_name, **dtype_kwarg(dtype), **LOAD_KWARGS.get(MODEL_KEY, {})).to(device).eval()
system = SYSTEM_PRESETS.get(SYSTEM, SYSTEM)
print('model      :', hf_name, '| revision', model_revision(model, hf_name), '| dtype', dtype, '| device', device)
print('format     :', fmt, '| system message:', repr(system), '| template supports system role:', supports_system(tok))
print('special    : bos', repr(tok.bos_token), tok.bos_token_id, '| eos', repr(tok.eos_token), tok.eos_token_id, '| pad', repr(tok.pad_token))
form = LETTER_FORMS[fmt][0]
ids = letter_ids(tok, form)
for L in LETTERS:
    print(f'letter {L}: scored token {form.format(L)!r} -> id {ids[L]} -> decodes back to {tok.decode([ids[L]])!r}')

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


model      : meta-llama/Llama-3.2-3B-Instruct | revision 0cb88a4f764b7a12671c53f0838cd831a0843b95 | dtype torch.float16 | device cuda
format     : chat | system message: None | template supports system role: True
special    : bos '<|begin_of_text|>' 128000 | eos '<|eot_id|>' 128009 | pad None
letter A: scored token 'A' -> id 32 -> decodes back to 'A'
letter B: scored token 'B' -> id 33 -> decodes back to 'B'
letter C: scored token 'C' -> id 34 -> decodes back to 'C'


## Step 4 - the exact input the model receives
`repr` makes newlines and special tokens visible. The table lists the last tokens
before the answer position: the model's prediction for the *next* token is what we score.

In [6]:
it = show[0]
text, off = build_text(tok, it['prompt'], fmt, None, system, TEMPLATE_KWARGS.get(MODEL_KEY))
enc_ids = tok(text, add_special_tokens=False)['input_ids']
print('FULL MODEL INPUT (repr):'); print(repr(text))
print(f'\n{len(enc_ids)} tokens | dataset prompt starts at char {off} | BOS tokens in input: {enc_ids.count(tok.bos_token_id)}')
print('decode(encode(text)) == text:', tok.decode(enc_ids) == text)
print('\nfirst 12 tokens:'); print(pd.DataFrame({'id': enc_ids[:12], 'token': [repr(tok.decode([t])) for t in enc_ids[:12]]}).T.to_string())
print('\nlast 25 tokens (the model predicts what comes AFTER the last one):')
print(pd.DataFrame({'id': enc_ids[-25:], 'token': [repr(tok.decode([t])) for t in enc_ids[-25:]]}).to_string())

FULL MODEL INPUT (repr):
"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 Jul 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nSolve the initial value problem below and choose the correct solution.\n\nDifferential equation: y' = -exp(x)\nConditions:\n1. y(1) = -4 - exp(1)\n2. z(1) = -exp(1) - 1\n\nOptions:\nA) y = -exp(x) - 4\nB) y = -exp(x) - 1\nC) y = 1 - exp(x)\nAnswer with the letter of the correct option only.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

137 tokens | dataset prompt starts at char 176 | BOS tokens in input: 1
decode(encode(text)) == text: True

first 12 tokens:
                        0                      1         2                    3       4      5       6             7        8    9            10   11
id                  128000                 128006      9125               128007     271  38766    1303         33025     2696   25         6790  220
token  '<|beg

## Step 5 - next-token scores, one item at a time (no batching, no padding)
This is the simplest possible scoring: one forward pass, read the next-token
distribution, compare the log-probs of the three letter tokens.

In [7]:
@torch.no_grad()
def score_one(prompt_text):
    t, _ = build_text(tok, prompt_text, fmt, None, system, TEMPLATE_KWARGS.get(MODEL_KEY))
    inp = tok(t, return_tensors='pt', add_special_tokens=False).to(device)
    logp = torch.log_softmax(model(**inp).logits[0, -1].float(), -1)
    lp = {L: logp[ids[L]].item() for L in LETTERS}
    top = torch.topk(logp, 10)
    return lp, [(tok.decode([i]), math.exp(v)) for v, i in zip(top.values.tolist(), top.indices.tolist())]

single = {}
for k, it in enumerate(items):
    lp, top = score_one(it['prompt'])
    z = sum(math.exp(v) for v in lp.values())
    single[it['id']] = {'logprobs': lp, 'pred': letter_from_scores(lp), 'letter_mass': z, 'top10': top}
    if it in show:
        r = single[it['id']]
        print('=' * 100); print(it['id'], it['variant'], it['subtype'], '| gold', it['answer'])
        print('  top-10 next tokens:', ', '.join(f'{t!r} {p:.3f}' for t, p in top))
        for L in LETTERS:
            print(f'  {L}: log p = {lp[L]:8.3f}   p = {math.exp(lp[L]):.4f}   p among A/B/C = {math.exp(lp[L]) / z:.3f}'
                  + ('   <- gold' if L == it['answer'] else ''))
        print(f'  letter mass {z:.3f} | PREDICTION {r["pred"]} | {"CORRECT" if r["pred"] == it["answer"] else "wrong"}')
acc1 = sum(single[i['id']]['pred'] == i['answer'] for i in items) / len(items)
print(f'\nunbatched accuracy on all {len(items)} {SPLIT} items: {acc1:.4f}')
print('predicted letters:', {L: sum(r['pred'] == L for r in single.values()) for L in LETTERS})

ivp-0000 B unrelated | gold A
  top-10 next tokens: 'To' 0.915, 'The' 0.064, 'B' 0.012, 'Since' 0.002, 'A' 0.001, 'Given' 0.001, 'C' 0.001, 'Option' 0.001, 'Int' 0.001, 'None' 0.000
  A: log p =   -6.542   p = 0.0014   p among A/B/C = 0.102   <- gold
  B: log p =   -4.417   p = 0.0121   p among A/B/C = 0.850
  C: log p =   -7.292   p = 0.0007   p among A/B/C = 0.048
  letter mass 0.014 | PREDICTION B | wrong
ivp-0001 B unrelated | gold A
  top-10 next tokens: 'C' 0.317, 'The' 0.239, 'B' 0.218, 'A' 0.201, 'To' 0.019, 'Since' 0.002, 'Option' 0.001, 'D' 0.000, 'Given' 0.000, 'After' 0.000
  A: log p =   -1.603   p = 0.2013   p among A/B/C = 0.274   <- gold
  B: log p =   -1.525   p = 0.2177   p among A/B/C = 0.296
  C: log p =   -1.150   p = 0.3167   p among A/B/C = 0.430
  letter mass 0.736 | PREDICTION C | wrong
ivp-0002 B duplicate | gold A
  top-10 next tokens: 'To' 0.974, 'The' 0.012, 'B' 0.007, 'Since' 0.003, 'Int' 0.001, 'A' 0.001, 'Given' 0.000, 'Option' 0.000, 'C' 0.000, 'None' 0

## Step 6 - the production script on the same items
Runs `baseline_letter_logprobs.py` exactly as the baseline does (batched, left
padding) into a scratch folder, then compares it item by item with step 5.

In [8]:
out = '/content/diag_runs' if IN_COLAB else 'diag_runs'
cmd = [sys.executable, 'baseline_letter_logprobs.py', '--model', MODEL_KEY, '--split', SPLIT, '--outdir', out]
if SYSTEM: cmd += ['--system', SYSTEM]
r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-2000:])
run_dir = sorted(glob.glob(os.path.join(out, f'*_{MODEL_KEY}*_{SPLIT}')))[-1]
batched = {row['id']: row for row in map(json.loads, open(os.path.join(run_dir, 'predictions.jsonl')))}
diff = max(abs(batched[i]['logprobs'][L] - single[i]['logprobs'][L]) for i in single for L in LETTERS)
agree = sum(batched[i]['pred'] == single[i]['pred'] for i in single)
print(f'max |log p difference| batched vs. one-by-one: {diff:.2e}   (fp16 noise is ~1e-2 or less)')
print(f'same prediction on {agree}/{len(single)} items')
print('script summary accuracy:', json.load(open(os.path.join(run_dir, 'summary.json')))['accuracy'])

llama-3.2-3b-instruct [dev, n=72] acc=0.3194  -> /content/diag_runs/20261102T165244Z_llama-3.2-3b-instruct_dev
max |log p difference| batched vs. one-by-one: 1.86e-01   (fp16 noise is ~1e-2 or less)
same prediction on 72/72 items
script summary accuracy: 0.3194


## Step 7 - what the model actually says (free generation, greedy)
The same input, but the model writes freely. `extract_letter` reads a letter from
the text; this is compared with the letter-score prediction from step 5.

In [9]:
@torch.no_grad()
def generate(prompt_text, max_new):
    t, _ = build_text(tok, prompt_text, fmt, None, system, TEMPLATE_KWARGS.get(MODEL_KEY))
    inp = tok(t, return_tensors='pt', add_special_tokens=False).to(device)
    g = model.generate(**inp, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(g[0, inp['input_ids'].shape[1]:], skip_special_tokens=True)

gen_rows = []
for it in show:
    txt = generate(it['prompt'], GEN_TOKENS)
    m = re.findall(r'\b(?:answer|option)\s*(?:is|:)?\s*\(?\**([ABC])\b', txt, re.I)
    parsed = extract_letter(txt.strip().splitlines()[0] if txt.strip() else '') or (m[-1].upper() if m else None)
    gen_rows.append({'id': it['id'], 'gold': it['answer'], 'letter_score_pred': single[it['id']]['pred'], 'generated_letter': parsed})
    print('=' * 100); print(it['id'], it['variant'], it['subtype'], '| gold', it['answer'], '| letter-score prediction', single[it['id']]['pred'])
    print('-' * 100); print(txt); print('-' * 100); print('letter read from the text:', parsed)
print(); print(pd.DataFrame(gen_rows).to_string(index=False))

ivp-0000 B unrelated | gold A | letter-score prediction B
----------------------------------------------------------------------------------------------------
To solve the initial value problem, we need to find the general solution of the differential equation and then apply the initial condition.

The general solution of y' = -exp(x) is:

y = ∫(-exp(x)) dx
y = ∫(-e^x) dx
y = -∫e^x dx
y = -e^x + C

Now, we apply the initial condition y(1) = -4 - exp(1):

-4 - exp(1) = -e^1 + C
-4 - e = -e + C
C = -4 - e + e
C = -4

So, the particular solution is:

y = -e^x - 4

Comparing this solution with the given options, we can see that:

The correct option is A) y = -exp(x) - 4.
----------------------------------------------------------------------------------------------------
letter read from the text: A
ivp-0001 B unrelated | gold A | letter-score prediction C
----------------------------------------------------------------------------------------------------
C
---------------------------------

## Step 8 - what goes into the grader, and what comes out

In [10]:
preds = [single[i['id']]['pred'] for i in items]
table = pd.DataFrame({'id': [i['id'] for i in items], 'variant': [i['variant'] for i in items],
                      'subtype': [i['subtype'] for i in items], 'gold': [i['answer'] for i in items],
                      'pred (passed to grader)': preds,
                      'p(pred) among A/B/C': [round(math.exp(single[i['id']]['logprobs'][p]) / single[i['id']]['letter_mass'], 3)
                                              for i, p in zip(items, preds)]})
table['correct'] = table['gold'] == table['pred (passed to grader)']
print(table.to_string(index=False))
acc, report = evaluate(items, preds)
print(f'\ngrader: evaluate(items, preds) -> accuracy {acc}')
print(f'manual recount: {table.correct.sum()}/{len(table)} = {table.correct.mean():.4f}')
print('baselines on this split: ' + ', '.join(f'always-{L} {evaluate(items, [L] * len(items))[0]:.3f}' for L in LETTERS))
print('\nconfusion (rows = gold, columns = predicted):'); print(pd.crosstab(table.gold, table['pred (passed to grader)']))
print('\nper-slice report:'); print(pd.DataFrame(report).T.to_string())

      id variant         subtype gold pred (passed to grader)  p(pred) among A/B/C  correct
ivp-0000       B       unrelated    A                       B                0.850    False
ivp-0001       B       unrelated    A                       C                0.430    False
ivp-0002       B       duplicate    A                       B                0.889    False
ivp-0003       B       unrelated    C                       C                0.556     True
ivp-0004       B       duplicate    B                       B                0.498     True
ivp-0005       B        rescaled    C                       B                0.702    False
ivp-0006       B        rescaled    A                       A                0.669     True
ivp-0007       B       unrelated    A                       A                0.754     True
ivp-0008       B       duplicate    A                       A                0.616     True
ivp-0009       B        rescaled    C                       A                0.4

## Step 9 - control: easy questions in the same format
If the model gets these right, the scoring pipeline can read correct answers out of it.

In [11]:
EASY = [("What is 2 + 2?", ["3", "4", "5"], "B"), ("What is the capital of France?", ["Paris", "Berlin", "Madrid"], "A"),
        ("Which number is even?", ["7", "9", "10"], "C"), ("What is 10 - 3?", ["7", "8", "6"], "A"),
        ("Which animal says 'meow'?", ["dog", "cat", "cow"], "B"), ("What is 3 * 3?", ["6", "12", "9"], "C"),
        ("What is the derivative of x^2?", ["x", "2*x", "x^3/3"], "B"), ("If y = 5*x, what is y(1)?", ["1", "0", "5"], "C"),
        ("Which is a colour?", ["blue", "table", "seven"], "A")]
ok = 0
for q, opts, gold in EASY:
    p = (f"Answer the question below.\n\n{q}\n\nOptions:\n" + '\n'.join(f'{L}) {o}' for L, o in zip(LETTERS, opts))
         + "\nAnswer with the letter of the correct option only.")
    lp, top = score_one(p); pred = letter_from_scores(lp); ok += pred == gold
    print(f'{q:35s} gold {gold} pred {pred}  top tokens: ' + ', '.join(f'{t!r} {pr:.2f}' for t, pr in top[:3]))
print(f'easy-question accuracy: {ok}/{len(EASY)}')

What is 2 + 2?                      gold B pred B  top tokens: 'B' 1.00, '.B' 0.00, 'A' 0.00
What is the capital of France?      gold A pred A  top tokens: 'A' 1.00, 'А' 0.00, "'A" 0.00
Which number is even?               gold C pred C  top tokens: 'C' 1.00, 'A' 0.00, 'B' 0.00
What is 10 - 3?                     gold A pred C  top tokens: 'C' 0.98, 'A' 0.02, 'B' 0.00
Which animal says 'meow'?           gold B pred B  top tokens: 'B' 1.00, '.B' 0.00, 'Б' 0.00
What is 3 * 3?                      gold C pred C  top tokens: 'C' 1.00, 'A' 0.00, 'B' 0.00
What is the derivative of x^2?      gold B pred B  top tokens: 'B' 0.83, 'C' 0.17, 'A' 0.00
If y = 5*x, what is y(1)?           gold C pred C  top tokens: 'C' 0.99, 'A' 0.01, 'B' 0.00
Which is a colour?                  gold A pred A  top tokens: 'A' 1.00, 'B' 0.00, 'C' 0.00
easy-question accuracy: 8/9


## Step 10 - does the choice follow the content or the letter?
Each item is shown 3 times with its options rotated (A,B,C → B,C,A → C,A,B), in
memory only (the dataset files are not touched). A model that reasons picks the
same **option text** each time; a model with letter bias picks the same **letter**.

In [12]:
def rotated_prompt(it, shift):
    p, spans = it['prompt'], it['spans']['options']
    texts = [o['text'] for o in it['options']]
    new = texts[shift:] + texts[:shift]
    for (s, e), t in sorted(zip(spans, new), reverse=True):
        p = p[:s] + t + p[e:]
    gold_text = it['answer_text']
    return p, LETTERS[new.index(gold_text)]

rows = []
for it in items:
    picks_text, picks_letter, corr = [], [], 0
    for shift in range(3):
        p, gold = rotated_prompt(it, shift)
        lp, _ = score_one(p); pred = letter_from_scores(lp)
        texts = [o['text'] for o in it['options']]; new = texts[shift:] + texts[:shift]
        picks_text.append(new[LETTERS.index(pred)]); picks_letter.append(pred); corr += pred == gold
    rows.append({'id': it['id'], 'variant': it['variant'], 'letters picked': ''.join(picks_letter),
                 'same option text all 3 times': len(set(picks_text)) == 1,
                 'same letter all 3 times': len(set(picks_letter)) == 1, 'correct (of 3)': corr})
rot = pd.DataFrame(rows)
print(rot.head(N_SHOW * 2).to_string(index=False))
print(f'\nsame option TEXT under all 3 orders : {rot["same option text all 3 times"].mean():.2%}  (content-driven)')
print(f'same LETTER under all 3 orders      : {rot["same letter all 3 times"].mean():.2%}  (position-driven)')
print(f'accuracy over the 3 orders          : {rot["correct (of 3)"].sum() / (3 * len(rot)):.4f}')
print(f'correct in all 3 orders             : {(rot["correct (of 3)"] == 3).mean():.2%} of items  (chance: 3.7%)')

      id variant letters picked  same option text all 3 times  same letter all 3 times  correct (of 3)
ivp-0000       B            BCB                         False                    False               2
ivp-0001       B            CCB                         False                    False               2
ivp-0002       B            BAC                          True                    False               0
ivp-0003       B            CBB                         False                    False               2
ivp-0004       B            BAC                          True                    False               3
ivp-0005       B            BBC                         False                    False               1
ivp-0006       B            ACA                         False                    False               2
ivp-0007       B            AAA                         False                     True               1
ivp-0008       B            AAA                         False            